# Model xatolarini tahlil qilish va zaxira nusxa yaratish

Fayl: `05_analyze_errors_and_backup_model.ipynb`. Ishga tushirish va papkalar izohi: [README.md](README.md).


In [1]:
from pathlib import Path

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

for folder in ["checkpoints", "models", "processor"]:
    path = PROJECT_DIR / folder

    print(f"\n📁 {folder}/")

    if path.exists():
        for item in sorted(path.rglob("*")):
            if item.is_file():
                size_mb = item.stat().st_size / (1024 ** 2)
                print(f"  {item.relative_to(path)} — {size_mb:.2f} MB")
    else:
        print("❌ Papka mavjud emas")


📁 checkpoints/
  full_lora_step_100/adapter_config.json — 0.00 MB
  full_lora_step_100/processor_config.json — 0.00 MB
  full_lora_step_100/tokenizer_config.json — 0.00 MB
  full_lora_step_100/training_state.pt — 9.55 MB
  full_lora_step_100/vocab.json — 0.00 MB
  full_lora_step_1000/adapter_config.json — 0.00 MB
  full_lora_step_1000/processor_config.json — 0.00 MB
  full_lora_step_1000/tokenizer_config.json — 0.00 MB
  full_lora_step_1000/training_state.pt — 9.61 MB
  full_lora_step_1000/vocab.json — 0.00 MB
  full_lora_step_10000/adapter_config.json — 0.00 MB
  full_lora_step_10000/processor_config.json — 0.00 MB
  full_lora_step_10000/tokenizer_config.json — 0.00 MB
  full_lora_step_10000/training_state.pt — 10.23 MB
  full_lora_step_10000/vocab.json — 0.00 MB
  full_lora_step_10500/adapter_config.json — 0.00 MB
  full_lora_step_10500/processor_config.json — 0.00 MB
  full_lora_step_10500/tokenizer_config.json — 0.00 MB
  full_lora_step_10500/training_state.pt — 10.27 MB
  full_lo

In [2]:
from pathlib import Path
import re
import torch

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

# Barcha checkpointlarni topamiz
checkpoints = []

for path in CHECKPOINT_DIR.glob("full_lora_step_*"):
    match = re.search(r"step_(\d+)$", path.name)

    if match:
        step = int(match.group(1))
        checkpoints.append((step, path))

checkpoints.sort(key=lambda x: x[0])

print("📁 Checkpointlar:")
for step, path in checkpoints:
    print(f"Step {step}: {path.name}")

# Oxirgi checkpoint
last_step, last_checkpoint = checkpoints[-1]

print("\n✅ Eng oxirgi checkpoint:")
print("Step:", last_step)
print("Path:", last_checkpoint)

# Checkpoint tarkibini tekshiramiz
state_path = last_checkpoint / "training_state.pt"

state = torch.load(
    state_path,
    map_location="cpu",
    weights_only=True
)

print("\n📦 Checkpoint tarkibi:")

for key, value in state.items():
    if isinstance(value, dict):
        print(f"{key}: dict ({len(value)} elements)")
    elif isinstance(value, torch.Tensor):
        print(f"{key}: Tensor {tuple(value.shape)}")
    else:
        print(f"{key}: {type(value).__name__}")

📁 Checkpointlar:
Step 50: full_lora_step_50
Step 100: full_lora_step_100
Step 200: full_lora_step_200
Step 300: full_lora_step_300
Step 400: full_lora_step_400
Step 500: full_lora_step_500
Step 750: full_lora_step_750
Step 1000: full_lora_step_1000
Step 1250: full_lora_step_1250
Step 1500: full_lora_step_1500
Step 1750: full_lora_step_1750
Step 2000: full_lora_step_2000
Step 2250: full_lora_step_2250
Step 2500: full_lora_step_2500
Step 2750: full_lora_step_2750
Step 3000: full_lora_step_3000
Step 3250: full_lora_step_3250
Step 3500: full_lora_step_3500
Step 3750: full_lora_step_3750
Step 4000: full_lora_step_4000
Step 4250: full_lora_step_4250
Step 4341: full_lora_step_4341
Step 4500: full_lora_step_4500
Step 5000: full_lora_step_5000
Step 5500: full_lora_step_5500
Step 6000: full_lora_step_6000
Step 6500: full_lora_step_6500
Step 7000: full_lora_step_7000
Step 7500: full_lora_step_7500
Step 8000: full_lora_step_8000
Step 8500: full_lora_step_8500
Step 8682: full_lora_step_8682
Step 90

In [3]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

files = [
    "results/test_step13023_greedy_metrics.csv",
    "results/test_greedy_vs_kenlm_metrics.csv",
    "results/validation_greedy_vs_kenlm_metrics.csv"
]

for file in files:
    path = PROJECT_DIR / file

    print(f"\n{'=' * 60}")
    print(file)

    if path.exists():
        df = pd.read_csv(path)
        display(df)
    else:
        print("Fayl topilmadi")


results/final_test_metrics.csv


,step,samples,loss,WER,CER,blank_ratio
0,13023,965,0.470927,49.142356,11.567896,53.487005



results/kenlm_final_test_metrics.csv


,Decoder,WER (%),CER (%)
0,Greedy,49.142356,11.567896
1,Beam Search + KenLM,27.421870,10.077606



results/kenlm_validation_comparison.csv


,Decoder,WER,CER
0,Greedy,49.889618,11.525404
1,KenLM_WER,26.904097,9.950211
2,KenLM_CER,27.705993,9.083989


In [4]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

pred_path = PROJECT_DIR / "results/test_greedy_vs_kenlm_predictions.csv"

df = pd.read_csv(pred_path)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

display(df.head(5))

Dataset shape: (965, 3)

Columns: ['reference', 'greedy', 'beam_kenlm']


,reference,greedy,beam_kenlm
0,paytaxtımızdıń kongressorayında mámleketimiz b...,paytaqtımıztıń kongres orayında mámleketimis b...,paytaxtımızdıń kongress orayında mámleketimiz ...
1,sanlı texnologiyalar baǵdarındaǵı ózara sóyles...,sanlı texnologiyalar baǵdarındaǵı óz ara sóles...,sanlı texnologiyalar baǵdarındaǵı óz ara sóyle...
2,jaslar hawazı festivalınıń ashılıw saltanatı b...,jaslar xawazı festivaldıń ashılı saltanadı bol...,jaslar hawazı festivaldıń ashılısaltanadı bolı...
3,sonıń menen birge ayırım máselelerdi tereńirek...,sonıń menen birge ayrı máselelerdi tereńreik ú...,sonıń menen birge ayırım máselelerdi tereńirek...
4,sońǵı jıllarda bilimlendiriw tarawında ámelge ...,soǵı jıllarda bililendiriw tarawında ámelge as...,sońǵı jıllarda bilimlendiriw tarawında ámelge ...


In [5]:
from collections import Counter
from jiwer import process_words

# KenLM predictionlarini tahlil qilamiz
references = df["reference"].fillna("").astype(str).tolist()
predictions = df["beam_kenlm"].fillna("").astype(str).tolist()

substitutions = Counter()
deletions = Counter()
insertions = Counter()

for ref, hyp in zip(references, predictions):

    result = process_words(ref, hyp)

    ref_words = result.references[0]
    hyp_words = result.hypotheses[0]

    for chunk in result.alignments[0]:

        if chunk.type == "substitute":
            ref_segment = ref_words[chunk.ref_start_idx:chunk.ref_end_idx]
            hyp_segment = hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx]

            for correct, predicted in zip(ref_segment, hyp_segment):
                substitutions[(correct, predicted)] += 1

        elif chunk.type == "delete":
            for word in ref_words[chunk.ref_start_idx:chunk.ref_end_idx]:
                deletions[word] += 1

        elif chunk.type == "insert":
            for word in hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx]:
                insertions[word] += 1

print("🔴 TOP 20 SUBSTITUTIONS")
for (correct, predicted), count in substitutions.most_common(20):
    print(f"{correct:20} → {predicted:20} : {count}")

print("\n🟠 TOP 20 DELETIONS")
for word, count in deletions.most_common(20):
    print(f"{word:20} : {count}")

print("\n🟡 TOP 20 INSERTIONS")
for word, count in insertions.most_common(20):
    print(f"{word:20} : {count}")

🔴 TOP 20 SUBSTITUTIONS
haqqında             → haqqındaǵı           : 24
jıyırma              → jigirma              : 16
sanaat               → sanat                : 10
ornında              → orında               : 8
elli                 → eliw                 : 8
joqarǵı              → joqarı               : 6
qırq                 → qırıq                : 6
atırǵanı             → atırǵan              : 5
sondayaq             → sonday               : 5
jılı                 → jıl                  : 5
altı                 → altınshı             : 5
ózbekstannıń         → ózbekstan            : 5
atırǵanın            → atırǵan              : 5
arasındaǵı           → arasında             : 5
olardıń              → olardı               : 4
processlerin         → proceslerin          : 4
ótilgenindey         → de                   : 4
júz                  → bir                  : 4
hám                  → menen                : 4
menen                → penen                : 4

🟠 TOP 20 DELE

In [6]:
from jiwer import process_words
import pandas as pd
from collections import Counter

def get_substitutions(references, predictions):
    substitutions = Counter()

    for ref, pred in zip(references, predictions):
        result = process_words(str(ref), str(pred))

        ref_words = result.references[0]
        pred_words = result.hypotheses[0]

        for chunk in result.alignments[0]:
            if chunk.type == "substitute":
                r = ref_words[chunk.ref_start_idx:chunk.ref_end_idx]
                p = pred_words[chunk.hyp_start_idx:chunk.hyp_end_idx]

                for a, b in zip(r, p):
                    substitutions[(a, b)] += 1

    return substitutions


greedy_errors = get_substitutions(
    df["reference"].fillna(""),
    df["greedy"].fillna("")
)

kenlm_errors = get_substitutions(
    df["reference"].fillna(""),
    df["beam_kenlm"].fillna("")
)

# KenLMda eng ko'p uchraydigan xatolar
rows = []

for (correct, predicted), count in kenlm_errors.most_common(20):
    rows.append({
        "Reference": correct,
        "Prediction": predicted,
        "Greedy_count": greedy_errors[(correct, predicted)],
        "KenLM_count": count
    })

comparison_df = pd.DataFrame(rows)

display(comparison_df)

,Reference,Prediction,Greedy_count,KenLM_count
0,haqqında,haqqındaǵı,21,24
1,jıyırma,jigirma,4,16
2,sanaat,sanat,9,10
3,ornında,orında,6,8
4,elli,eliw,7,8
5,joqarǵı,joqarı,4,6
6,qırq,qırıq,4,6
7,atırǵanı,atırǵan,5,5
8,sondayaq,sonday,4,5
9,jılı,jıl,6,5


In [7]:
from pathlib import Path
from collections import Counter

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")
KENLM_DIR = PROJECT_DIR / "kenlm"

# KenLM papkasidagi fayllarni aniqlash
for path in KENLM_DIR.rglob("*"):
    if path.is_file():
        print(
            path.relative_to(KENLM_DIR),
            f"{path.stat().st_size / 1024**2:.2f} MB"
        )

karakalpak_5gram.arpa 89.39 MB
karakalpak_5gram.binary 36.87 MB
decoder_config.json 0.00 MB
karakalpak_train.txt 4.99 MB
kenlm_source/README.md 0.01 MB
kenlm_source/compile_query_only.sh 0.00 MB
kenlm_source/CMakeLists.txt 0.00 MB
kenlm_source/.gitignore 0.00 MB
kenlm_source/COPYING.3 0.03 MB
kenlm_source/COPYING.LESSER.3 0.01 MB
kenlm_source/setup.py 0.00 MB
kenlm_source/MANIFEST.in 0.00 MB
kenlm_source/clean_query_only.sh 0.00 MB
kenlm_source/pyproject.toml 0.00 MB
kenlm_source/BUILDING 0.00 MB
kenlm_source/Doxyfile 0.06 MB
kenlm_source/COPYING 0.03 MB
kenlm_source/LICENSE 0.00 MB
kenlm_source/build/CMakeCache.txt 0.02 MB
kenlm_source/build/Makefile 0.01 MB
kenlm_source/build/kenlmConfig.cmake 0.00 MB
kenlm_source/build/cmake_install.cmake 0.02 MB
kenlm_source/build/lib.linux-aarch64-cpython-311/libkenlm.so 0.76 MB
kenlm_source/build/lib.linux-aarch64-cpython-311/kenlm.cpython-311-aarch64-linux-gnu.so 0.86 MB
kenlm_source/build/util/Makefile 0.04 MB
kenlm_source/build/util/cmake_inst

In [8]:
from pathlib import Path
from collections import Counter

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

CORPUS_PATH = PROJECT_DIR / "kenlm/karakalpak_train.txt"

# KenLM korpusini o'qish
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Jami gaplar:", len(lines))

# Barcha so'zlarning chastotasi
word_counts = Counter()

for line in lines:
    word_counts.update(line.lower().split())

# Tekshiriladigan so'zlar
words = [
    "jıyırma",
    "jigirma",
    "sanaat",
    "sanat",
    "joqarǵı",
    "joqarı",
    "haqqında",
    "haqqındaǵı",
    "elli",
    "eliw",
    "júz",
    "bir",
    "hám",
    "menen"
]

print("\nKENLM CORPUS WORD FREQUENCY")
print("-" * 45)

for word in words:
    print(f"{word:<20} : {word_counts[word]}")

Jami gaplar: 17362

KENLM CORPUS WORD FREQUENCY
---------------------------------------------
jıyırma              : 821
jigirma              : 1957
sanaat               : 311
sanat                : 1
joqarǵı              : 264
joqarı               : 1479
haqqında             : 1116
haqqındaǵı           : 478
elli                 : 708
eliw                 : 3
júz                  : 2723
bir                  : 3821
hám                  : 18809
menen                : 4892


In [9]:
from pathlib import Path
import random

CORPUS_PATH = Path(
    "/home/intern/projects/karakalpak_stt/kenlm/karakalpak_train.txt"
)

with open(CORPUS_PATH, encoding="utf-8") as f:
    lines = f.readlines()

words = ["jıyırma", "jigirma", "sanaat", "sanat"]

random.seed(42)

for word in words:
    matches = [
        line.strip()
        for line in lines
        if word in line.lower().split()
    ]

    print(f"\n{'=' * 65}")
    print(f"WORD: {word}")
    print(f"Sentences: {len(matches)}")

    for sentence in random.sample(matches, min(5, len(matches))):
        print("→", sentence[:250])


WORD: jıyırma
Sentences: 721
→ b mámleketlik mákemelerde ayaq oyın kórkem óneri tarawında jumıs alıp barıp atırǵan ayaq oyınshılardı jeńilletilgen jumıs stajı menen pensiyaǵa shıǵarıwda talap etiletuǵın jıyırma jıllıq jumıs stajınıń keminde on bes jılı ayaq oyınshılıq jumıstan iba
→ usı tárizde jıyırma úsh jergilikli brend awǵanstan bazarına kirip bardı awǵanstan tájikstan ázerbayjan qazaqstan pakistan hám qırǵızstanda sawda úyleri shourum hám sawda qoymaxanaları shólkemlestirilip elli den aslam túrdegi joqarı qosımsha qunlı eks
→ atap aytqanda samarqand sırdárya hám jizzaqtaǵı úyreniwlerde segiz júz jetpis million kilovatt saat elektr hám tórt júz jıyırma million kub metr gazdi únemlewdiń jańa imkaniyatları anıqlandı
→ búgingi kúnde mámleketlik xızmetlerge bolǵan talaptıń joqarılıǵı reformalardıń áhmiyetliligin kórsetpekte atap aytqanda eki mıń jigirma altınshı jıldıń basınan berli my gov uz portalı arqalı jıyırma tórt millionnan aslam xızmet kórsetilgen
→ sudyalar joqarı mektebine e

In [10]:
from pathlib import Path
import shutil
import zipfile

PROJECT = Path("/home/intern/projects/karakalpak_stt")
BACKUP = PROJECT / "karakalpak_stt_backup.zip"

# Saqlanadigan papkalar
folders = [
    "checkpoints",
    "models",
    "processor",
    "kenlm",
    "results"
]

# Notebook va kodlar
files = list(PROJECT.glob("*.ipynb")) + list(PROJECT.glob("*.py"))

with zipfile.ZipFile(
    BACKUP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in folders:
        folder_path = PROJECT / folder

        if not folder_path.exists():
            continue

        for file in folder_path.rglob("*"):
            if not file.is_file():
                continue

            # KenLM source/build fayllarini olmaymiz
            if "kenlm_source" in file.parts:
                continue

            archive.write(
                file,
                arcname=file.relative_to(PROJECT)
            )

    for file in files:
        archive.write(
            file,
            arcname=file.name
        )

print("✅ Backup tayyor!")
print("📁", BACKUP)
print("📦 Hajmi:", round(BACKUP.stat().st_size / 1024**2, 2), "MB")

✅ Backup tayyor!
📁 /home/intern/projects/karakalpak_stt/karakalpak_stt_backup.zip
📦 Hajmi: 475.06 MB


In [11]:
with zipfile.ZipFile(BACKUP, "r") as archive:
    bad_file = archive.testzip()

    print("Fayllar soni:", len(archive.namelist()))

    if bad_file is None:
        print("✅ ZIP arxiv tekshiruvdan o'tdi!")
    else:
        print("❌ Muammoli fayl:", bad_file)

Fayllar soni: 264
✅ ZIP arxiv tekshiruvdan o'tdi!
